In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from dotenv import load_dotenv
import joblib


In [2]:
DATASETS_PATH = "/home/mariya/Data/pallas/mlomix_datasets"

TRAIN_DATASET_PATH = os.path.join(DATASETS_PATH, "train-dnam")
X_train_path = os.path.join(TRAIN_DATASET_PATH, "features.dnam.csv")
y_train_path = os.path.join(TRAIN_DATASET_PATH, "labels.csv")

CALIBRATE_DATASET_PATH = os.path.join(DATASETS_PATH, "calibrate-dnam")
X_calibrate_path = os.path.join(CALIBRATE_DATASET_PATH, "features.dnam.csv")
y_calibrate_path = os.path.join(CALIBRATE_DATASET_PATH, "labels.csv")

TEST_DATASET_PATH = os.path.join(DATASETS_PATH, "test-dnam")
X_test_path = os.path.join(TEST_DATASET_PATH, "features.dnam.csv")
y_test_path = os.path.join(TEST_DATASET_PATH, "labels.csv")

output_dir = "/home/mariya/Data/pallas/models/dnam"
random_seed = 42
n_jobs = 10
rf_model_path = os.path.join(output_dir, "final_model.pkl")
selected_features_path = os.path.join(output_dir, "selected_features.txt")
pipeline_path = os.path.join(output_dir, "pipeline.pkl")
hyperparams_path = os.path.join(output_dir, "hyperparams.pkl")

# Print configuration
print(f"y_test_path: {y_test_path}")
# print(f"X_test_feature_selected_path: {X_test_feature_selected_path}")
print(f"output_dir: {output_dir}")
print(f"random_seed: {random_seed}")
print(f"n_jobs: {n_jobs}")
print(f"rf_model_path: {rf_model_path}")
print(f"selected_features_path: {selected_features_path}")
print(f"pipeline_path: {pipeline_path}")
print(f"hyperparams_path: {hyperparams_path}")

y_test_path: /home/mariya/Data/pallas/mlomix_datasets/test-dnam/labels.csv
output_dir: /home/mariya/Data/pallas/models/dnam
random_seed: 42
n_jobs: 10
rf_model_path: /home/mariya/Data/pallas/models/dnam/final_model.pkl
selected_features_path: /home/mariya/Data/pallas/models/dnam/selected_features.txt
pipeline_path: /home/mariya/Data/pallas/models/dnam/pipeline.pkl
hyperparams_path: /home/mariya/Data/pallas/models/dnam/hyperparams.pkl


In [3]:
X_test = pd.read_csv(X_test_path, index_col=0)
y_test = pd.read_csv(y_test_path, index_col=0)
X_cal = pd.read_csv(X_calibrate_path, index_col=0)
y_cal = pd.read_csv(y_calibrate_path, index_col=0)
X_test, y_test, X_cal, y_cal

(              cg00000029  cg00000236  cg00001269  cg00001793  cg00002769  \
 id                                                                         
 ALL_100           0.6424      0.9337      0.3086      0.0537      0.9014   
 ALL_1029          0.8833      0.9147      0.2128      0.0535      0.9578   
 ALL_1064          0.7359      0.8970      0.4739      0.4495      0.9185   
 ALL_1068          0.5814      0.8970      0.3979      0.1976      0.9577   
 ALL_1076          0.6039      0.6706      0.5073      0.3932      0.9561   
 ...                  ...         ...         ...         ...         ...   
 TCGA-AB-2896      0.5763      0.6832      0.8928      0.2707      0.9350   
 TCGA-AB-2900      0.5614      0.9418      0.9254      0.9343      0.9533   
 TCGA-AB-2919      0.7575      0.9237      0.9252      0.9380      0.9485   
 TCGA-AB-2973      0.8209      0.8511      0.9057      0.9089      0.9120   
 TCGA-AB-2992      0.9088      0.8856      0.8701      0.9128      0.8853   

In [4]:
# align the indices
X_test = X_test.loc[y_test.index]
X_cal = X_cal.loc[y_cal.index]
X_test, X_cal

(              cg00000029  cg00000236  cg00001269  cg00001793  cg00002769  \
 id                                                                         
 ALL_100           0.6424      0.9337      0.3086      0.0537      0.9014   
 ALL_1029          0.8833      0.9147      0.2128      0.0535      0.9578   
 ALL_1064          0.7359      0.8970      0.4739      0.4495      0.9185   
 ALL_1068          0.5814      0.8970      0.3979      0.1976      0.9577   
 ALL_1076          0.6039      0.6706      0.5073      0.3932      0.9561   
 ...                  ...         ...         ...         ...         ...   
 TCGA-AB-2896      0.5763      0.6832      0.8928      0.2707      0.9350   
 TCGA-AB-2900      0.5614      0.9418      0.9254      0.9343      0.9533   
 TCGA-AB-2919      0.7575      0.9237      0.9252      0.9380      0.9485   
 TCGA-AB-2973      0.8209      0.8511      0.9057      0.9089      0.9120   
 TCGA-AB-2992      0.9088      0.8856      0.8701      0.9128      0.8853   

In [5]:
y_test = y_test.iloc[:, 0].values # Convert to 1D numpy array
y_cal = y_cal.iloc[:, 0].values # Convert to 1D numpy array
y_test, y_cal

(array(['B-ALL, high hyperdiploid', 'B-ALL, high hyperdiploid',
        'B-ALL, high hyperdiploid', 'B-ALL, KMT2A-r', 'B-ALL, KMT2A-r',
        'B-ALL, PAX5alt', 'B-ALL, high hyperdiploid', 'B-ALL, KMT2A-r',
        'B-ALL, PAX5alt', 'B-ALL, high hyperdiploid', 'B-ALL, ETV6::RUNX1',
        'B-ALL, ETV6::RUNX1', 'B-ALL, ETV6::RUNX1',
        'B-ALL, high hyperdiploid', 'B-ALL, high hyperdiploid',
        'B-ALL, ETV6::RUNX1', 'B-ALL, ETV6::RUNX1',
        'B-ALL, high hyperdiploid', 'B-ALL, KMT2A-r',
        'B-ALL, high hyperdiploid', 'B-ALL, high hyperdiploid',
        'B-ALL, ETV6::RUNX1', 'B-ALL, ETV6::RUNX1', 'B-ALL, TCF3::PBX1',
        'B-ALL, high hyperdiploid', 'B-ALL, ETV6::RUNX1', 'B-ALL, PAX5alt',
        'B-ALL, ETV6::RUNX1', 'B-ALL, ETV6::RUNX1', 'B-ALL, ETV6::RUNX1',
        'B-ALL, high hyperdiploid', 'B-ALL, high hyperdiploid',
        'B-ALL, high hyperdiploid', 'B-ALL, high hyperdiploid',
        'B-ALL, ETV6::RUNX1', 'B-ALL, high hyperdiploid',
        'B-ALL, high 

In [6]:
# Merge test and cal
X_merged = pd.concat([X_cal, X_test], axis=0)
y_merged = np.concatenate([y_cal, y_test], axis=0)
X_merged, y_merged

(              cg00000029  cg00000236  cg00001269  cg00001793  cg00002769  \
 id                                                                         
 ALL_10            0.5230      0.9130      0.0743      0.1525      0.9553   
 ALL_1005          0.8918      0.9069      0.2387      0.7247      0.9514   
 ALL_1009          0.9171      0.9089      0.0931      0.0748      0.9669   
 ALL_1016          0.8869      0.9237      0.0526      0.1417      0.9629   
 ALL_1028          0.2356      0.9257      0.1341      0.0648      0.9583   
 ...                  ...         ...         ...         ...         ...   
 TCGA-AB-2896      0.5763      0.6832      0.8928      0.2707      0.9350   
 TCGA-AB-2900      0.5614      0.9418      0.9254      0.9343      0.9533   
 TCGA-AB-2919      0.7575      0.9237      0.9252      0.9380      0.9485   
 TCGA-AB-2973      0.8209      0.8511      0.9057      0.9089      0.9120   
 TCGA-AB-2992      0.9088      0.8856      0.8701      0.9128      0.8853   

In [7]:
print("\ny_test class distribution:")
print(y_merged.value_counts())


y_test class distribution:


AttributeError: 'numpy.ndarray' object has no attribute 'value_counts'

In [ ]:
# Load model
final_rf = joblib.load(rf_model_path)

# Load selected features from text file
selected_features = pd.read_csv(selected_features_path, header=None)[0].tolist()

# Use selected features for test set
y_pred_final = final_rf.predict(X_merged)

# Print size of selected features
print(f"Number of selected features: {len(selected_features)}")

# Print shape of X_test_selected
print(f"Shape of X_test_selected: {X_merged.shape}")

Number of selected features: 18191
Shape of X_test_selected: (726, 18191)


In [22]:
print("\nTest set classification report (final optimized model):")
print(classification_report(y_merged, y_pred_final, digits=3))



Test set classification report (final optimized model):
                          precision    recall  f1-score   support

      AML, APL-PML::RARA      1.000     0.857     0.923         7
     AML, CBFA2T3::GLIS2      1.000     0.800     0.889        10
        AML, CBFB::MYH11      0.964     0.981     0.972        54
           AML, CEBPAalt      1.000     0.957     0.978        23
        AML, DEK::NUP214      1.000     0.875     0.933         8
             AML, ETV6-r      0.857     0.857     0.857         7
           AML, FUS::ERG      1.000     1.000     1.000         2
            AML, KAT6A-r      1.000     0.333     0.500         3
            AML, KMT2A-r      0.806     0.952     0.873       105
            AML, MECOM-r      0.000     0.000     0.000         2
            AML, NPM1alt      0.940     0.839     0.887        56
            AML, NUP98-r      0.947     0.600     0.735        30
        AML, RBM15::MKL1      1.000     0.333     0.500         3
     AML, RUNX1::R

/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

In [23]:
# ----------------------
# Extract top cpg sites from final model
# ----------------------
# Get feature importances from final RF
importances = final_rf.feature_importances_

# Rank genes by importance
gene_importances = pd.DataFrame({
    "Cpg Site": selected_features,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

print("\nTop 20 cpg sites driving classification (final model):")
print(gene_importances.head(20))


Top 20 cpg sites driving classification (final model):
         Cpg Site  Importance
4857   cg06623668    0.012113
13     cg00015639    0.007003
5069   cg06891858    0.006603
476    cg00587115    0.006557
12850  cg18638389    0.005896
12074  cg17411792    0.004822
325    cg00424467    0.004547
14658  cg21912448    0.004507
5351   cg07232063    0.004505
6006   cg08166720    0.004129
10589  cg14992144    0.004095
1462   cg01912455    0.004031
17599  cg26786253    0.003830
16675  cg25287071    0.003634
8660   cg12183594    0.003531
301    cg00393348    0.003240
2983   cg03930153    0.003216
4508   cg06078469    0.003195
6226   cg08457158    0.003080
8733   cg12338894    0.003024


In [24]:
# How many of the feature importances are zero?
num_zero_importances = np.sum(importances == 0)
print(f"\nNumber of features with zero importance: {num_zero_importances} out of {len(importances)}")

# Print lowest importance genes as well
print("\nBottom 20 genes by importance (final model):")
print(gene_importances.tail(20))


Number of features with zero importance: 4997 out of 18191

Bottom 20 genes by importance (final model):
         Cpg Site  Importance
4584   cg06222851         0.0
4589   cg06237774         0.0
4591   cg06240450         0.0
4597   cg06248560         0.0
14105  cg20864608         0.0
14113  cg20879085         0.0
4566   cg06180869         0.0
4571   cg06188083         0.0
7972   cg11132534         0.0
4577   cg06202686         0.0
4580   cg06205746         0.0
14120  cg20891842         0.0
14091  cg20820063         0.0
14097  cg20842915         0.0
7959   cg11113760         0.0
7960   cg11116200         0.0
7964   cg11124364         0.0
7969   cg11130317         0.0
7971   cg11131532         0.0
4573   cg06191495         0.0
